In [2]:
import tensorflow as tf
import numpy as np 
from utils import * 
import keras as kr 

In [3]:
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y,R = load_ratings_small() # Y => Actual Rating ( if there is no rating it is considered to be 0 )
                           # R => Is rating exists ?
print(Y.shape)

(4778, 443)


In [4]:
ratings = Y[R == 1] # It stores those ratings in ratings array where that actually exist (R==1 means if rating exists)
mean = np.mean(ratings)

<h1> Cost Function (Standard)</h1>

In [5]:
def calc_cost(X, W, B, Y, R, lambda_):
    nm,nu  = Y.shape
    J = 0
    for j in range(nu):
        w = W[j,:] # select all colmns of row 1 
        b = B[0,j] # select i column of first row because it has only one rows
        for i in range(nm): 
            y = Y[i,j] # For movie (i) get the user (j) rating 
            r = R[i,j] # For movie (i) get the user (j) rating if exist 
            x = X[i,:] # For movie (i) get all the features like how much it is (romantic,action,thriller etc)
            J += r*(np.square(np.dot(w,x)+b-y)) # We are multiplying by r to full fill the "r(i,j)==1" condition of cost function formula means consider only the error for those where user j has rated the movie i  
    J+=(lambda_)*(np.sum(np.square(W))+np.sum(np.square(X)))
    J= J/2
    return J


<h1> Cost Function (vectorized)</h1>

In [6]:
def vec_cost (X, W, B, Y, R, lambda_ ):
    # We are multiplying by r to full fill the "r(i,j)==1" condition of cost function formula means consider only the error for those where user j has rated the movie i 
    J = ( tf.linalg.matmul(X,tf.transpose(W))+B - Y )*R # This returns an Array also we are taking transpose so that dimension should match for multiplication(AxB = BxA)
    regularization = (lambda_/2)*(tf.reduce_sum(W**2) + tf.reduce_sum(X**2))
    J = 0.5*(tf.reduce_sum(J**2)) + regularization
    return J

<h1>Rating the Movies</h1>

In [13]:
movieList, movieList_df = load_Movie_List_pd()
my_ratings = np.zeros(num_movies)# We are generating and np array of zeros of length equal to num of movies so that we have to rate only those movies which i like and the other movies rating will be zero automatically 

# Now we are rating the Movies we like by replacing the zero with our rating 

my_ratings[929]  = 2   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 5   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)

my_rated = [i for i in range(len(my_ratings)) if my_ratings[i]>0] # getting the index of movies which we rated and putting it inside array

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 2.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 5.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


<h1>Adding new column of my ratings in data set</h1>

In [ ]:
Y,R = load_ratings_small()
Y = np.c_[my_ratings,Y] # Add new column my_rating in the Y(all user rating) at start
R = np.c_[(my_ratings != 0 ).astype(int),R] # Adding column of (ones and zeros) in R that tells that did my rating exist for movie
# normalizing the data 
Ynorm, Ymean = normalizeRatings(Y, R)

(4778, 444)


<h1>Training model</h1>

In [ ]:
# maing tensorflow variables 
num_movies,num_users = Y.shape
num_features = 100
tf.random.set_seed("1234")
'''
Below the tf.random.normal(num_users,num_features) produce random " guassian (normal) distribution values " 
which are very small like 0.5,0.6 etc. We do this so that our starting parameters like weight and bias are
not too big like 500,300 etc which can effect our model learning
'''
W = tf.Variable(tf.random.normal(num_users,num_features),dtype=tf.d)

In [ ]:
def train_model(X, W, B, Ynorm, R, lambda_):

In [8]:
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
B_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]
cost1 = calc_cost(X_r,W_r,B_r,Y_r,R_r,0)
cost2 = vec_cost(X_r,W_r,B_r,Y_r,R_r,0)
print(f"Cost 1 : {cost1}\nCost 2 : {cost2} ")

Cost 1 : 13.670725805579915
Cost 2 : 13.670725805579917 
